In [1]:
import re
import spacy
import re
import pandas as pd

nlp = spacy.load("en_core_web_sm")

FOOD_STOPWORDS = {
    "cook", "heat", "minute", "hour", "time", "pan", "pot",
    "add", "mix", "stir", "serve", "taste", "water"
}

def extract_food_words(text):
    if not isinstance(text, str):
        return []

    text = re.sub(r'c\(|\)', '', text)

    doc = nlp(text)

    foods = []

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            word = token.lemma_.lower().strip()

            if word not in FOOD_STOPWORDS and len(word) > 2:
                foods.append(word)

    return list(set(foods))

In [2]:
sample = """
Add pork bones and peas.
Cook with oil and salt.
"""

extract_food_words(sample)

['salt', 'pork', 'pea', 'oil', 'bone']

In [3]:
recipes = pd.read_pickle("../data/candidate_recipes.pkl")

recipes["instruction_ingredients"] = recipes["RecipeInstructions"].apply(extract_food_words)

In [4]:
recipes["instruction_ingredients"] = recipes["RecipeInstructions"].apply(extract_food_words)

recipes["augmented_ingredients"] = recipes.apply(
    lambda row: list(set(row["clean_ingredients"]) | set(row["instruction_ingredients"])),
    axis=1
)

recipes.to_pickle("../data/final_nlp_recipes.pkl")

print("final_nlp_recipes.pkl saved successfully")

final_nlp_recipes.pkl saved successfully


In [5]:
# Build vocabulary of real ingredients
ingredient_vocab = set()

for ings in recipes["clean_ingredients"]:
    ingredient_vocab.update(ings)

len(ingredient_vocab)

2891

In [6]:
def extract_food_words(text):
    if not isinstance(text, str):
        return []

    text = re.sub(r'c\(|\)', '', text)
    doc = nlp(text)

    foods = []

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            word = token.lemma_.lower().strip()

            # ✅ keep only real ingredients
            if word in ingredient_vocab:
                foods.append(word)

    return list(set(foods))

In [7]:
recipes["instruction_ingredients"] = recipes["RecipeInstructions"].apply(extract_food_words)

recipes["augmented_ingredients"] = recipes.apply(
    lambda row: list(set(row["clean_ingredients"]) | set(row["instruction_ingredients"])),
    axis=1
)

recipes.to_pickle("../data/final_nlp_recipes2.pkl")